In [ ]:
from typing import Self

import pydantic

In [ ]:
import marimo as mo

In [ ]:
import os
from pathlib import Path
from ortools.sat.python import cp_model

# 資源制約付きプロジェクトスケジューリング問題 (CP-SAT)

定式化や HiGHS での求解は `resource_constrained.py` を参照.

## インスタンス

kobe-scheduling の patterson.rcp を使う. フォーマットは `resource_constrained.py` に書いた通り (多分).

In [ ]:
parent = str(Path(os.path.abspath(__file__)).parent)
data_dir = Path(parent, "kobe-scheduling", "data", "rcpsp", "patterson.rcp")

In [ ]:
class Job(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(frozen=True)
    id: int
    time: int
    res_usages: list[int]

In [ ]:
class Resource(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(frozen=True)
    ub: int

In [ ]:
class Condition(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(frozen=True)
    jobs: list[Job]
    ress: list[Resource]
    prec: set[tuple[int, int]]

    @classmethod
    def from_file(cls, filepath) -> Self:
        prec = set()
        with open(filepath) as f:
            _njobs, _nress = map(lambda s: int(s), f.readline().split())

            f.readline()

            resubs = list(map(lambda s: int(s), f.readline().split()))

            ress = [Resource(ub=resub) for resub in resubs]

            jobs = []
            job_id = 0
            while line := f.readline():
                datas = list(map(lambda s: int(s), line.split()))
                if len(datas) == 0:
                    continue

                idx = 0
                time = datas[idx]
                idx += 1
                res_usages = [datas[idx + jdx] for jdx in range(_nress)]
                idx += _nress

                # 次から始まる数値列の長さなのでスキップ
                idx += 1

                while idx < len(datas):
                    prec.add((job_id, datas[idx] - 1))
                    idx += 1

                jobs.append(Job(id=job_id, time=time, res_usages=res_usages))
                job_id += 1

        return cls(jobs=jobs, ress=ress, prec=prec)

In [ ]:
_filepath = data_dir / Path("pat1.rcp")
cond1 = Condition.from_file(_filepath)

## CP-SAT によるモデリング

In [ ]:
class Model2CpSat:
    def __init__(self, cond: Condition):
        self.model = cp_model.CpModel()

        horizon = sum(job.time for job in cond.jobs)

        self.starts = [
            self.model.new_int_var(lb=0, ub=horizon - job.time, name="")
            for job in cond.jobs
        ]
        self.jobs = [
            self.model.new_fixed_size_interval_var(
                self.starts[id_job], job.time, name=""
            )
            for id_job, job in enumerate(cond.jobs)
        ]

        # ジョブ間依存関係
        for idx, jdx in cond.prec:
            self.model.add(
                self.jobs[idx].end_expr() <= self.jobs[jdx].start_expr()
            )

        # 資源制約
        for id_res, res in enumerate(cond.ress):
            capacity = res.ub
            intervals = []
            demands = []
            for id_job, job in enumerate(cond.jobs):
                if job.res_usages[id_res] == 0:
                    continue
                intervals.append(self.jobs[id_job])
                demands.append(job.res_usages[id_res])

            self.model.add_cumulative(intervals, demands, capacity)

        # 目的関数: makespan
        self.objective = self.model.new_int_var(lb=0, ub=horizon, name="")
        self.model.add_max_equality(
            self.objective, [interval.end_expr() for interval in self.jobs]
        )
        self.model.minimize(self.objective)

    def solve(self, timeout: int = 180):
        self.solver = cp_model.CpSolver()
        self.solver.parameters.log_search_progress = True
        self.solver.parameters.max_time_in_seconds = timeout
        self.status = self.solver.solve(self.model)

In [ ]:
model4 = Model2CpSat(cond1)
model4.solve()


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 180 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0x274e8745b400f59d)
#Variables: 15 (#ints: 1 in objective) (14 primary variables)
  - 2 in [0,34]
  - 1 in [0,35]
  - 2 in [0,36]
  - 3 in [0,37]
  - 2 in [0,38]
  - 2 in [0,39]
  - 3 in [0,40]
#kCumulative: 3 (#intervals: 9)
#kInterval: 14
#kLinMax: 1 (#expressions: 14)
#kLinear2: 20

Starting presolve at 0.00s
  1.30e-05s  0.00e+00d  [DetectDominanceRelations] 
  5.42e-04s  1.60e-07d  [PresolveToFixPoint] #num_loops=8 #num_dual_strengthening=4 
  1.26e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  5.20e-06s  0.00e+00d  [DetectDuplicateColumns] 
  1.18e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 63 nodes and 75 arcs.
[Symmetry] Symmetry computation done. time: 1.5479e-05 dtime: 7.23e-06
  9.07e-06s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  8.

In [ ]:
print(f"Opt.value = {model4.solver.value(model4.objective)}")

for _id_job, interval in enumerate(model4.jobs):
    _val = model4.solver.value(interval.start_expr())
    print(f"s[{_id_job}] = {_val}")

Opt.value = 19
s[0] = 0
s[1] = 0
s[2] = 0
s[3] = 3
s[4] = 5
s[5] = 4
s[6] = 6
s[7] = 12
s[8] = 14
s[9] = 6
s[10] = 9
s[11] = 11
s[12] = 14
s[13] = 19


In [ ]:
_filepath = data_dir / Path("pat104.rcp")
cond2 = Condition.from_file(_filepath)

In [ ]:
model6 = Model2CpSat(cond2)
model6.solve()


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 180 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0xe4d929acad86d0b6)
#Variables: 52 (#ints: 1 in objective) (51 primary variables)
  - 2 in [0,180]
  - 2 in [0,181]
  - 6 in [0,183]
  - 11 in [0,184]
  - 2 in [0,185]
  - 9 in [0,186]
  - 12 in [0,187]
  - 5 in [0,188]
  - 3 in [0,189]
#kCumulative: 3 (#intervals: 147)
#kInterval: 51
#kLinMax: 1 (#expressions: 51)
#kLinear2: 81

Starting presolve at 0.00s
  1.96e-05s  0.00e+00d  [DetectDominanceRelations] 
  1.90e-03s  4.16e-07d  [PresolveToFixPoint] #num_loops=13 #num_dual_strengthening=2 
  2.19e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  1.44e-05s  0.00e+00d  [DetectDuplicateColumns] 
  3.14e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 430 nodes and 650 arcs.
[Symmetry] Symmetry computation done. time: 6.828e-05 dtime: 6.216e-05
  3.00e-05s  0.00e+00d  [DetectDupl